In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *
from prompts.synonym_context_prompt import *

load_dotenv()

pd.set_option('display.max_rows', None)
#"gpt-4.1-mini"
#"gpt-4o-mini"
#"gpt-4.1"
#"gpt-4o"

model_ = "gpt-5.1"
api_key = os.getenv("API_KEY")


llm = llm_call(model_version = model_, api_key= api_key)

In [2]:
# "Credit-TEST-POLLUTED.NORND-activity-0.3-0"
# "Credit-TEST-SYNONYM-0.3-0"
# "Credit-TRAIN-DISTORTED-activity-0.3-0"
#Pub-Collateral
#Credit-TRAIN-HOMONYM-0.3-0
# "Credit-TEST-CLEAN"
test =  "Credit-TRAIN-DISTORTED-activity-0.3-0"
LOG_NAME = f"./dataset/{test}.csv" 

df_new, cases_json = build_event_jsons(log_name = LOG_NAME, chunk_cases = 10)

#sample_size = int(len(df_new['case_id'].unique()) * 0.05)
#np.random.seed(42)
#selected_case_ids = np.random.choice(df_new['case_id'].unique(), size=sample_size, replace=False)
#df_new = df_new[df_new['case_id'].isin(selected_case_ids)].copy()


df_new.head(3)


,event_id,case_id,activity,timestamp,label
0,0,0,Check for completeness,2023-09-29 09:00:00.000,NaN
1,1,0,New online application received,2023-09-29 09:00:00.000,NaN
2,2,0,Perform checks,2023-09-29 09:08:36.418,NaN


In [3]:
act_freq_dict = df_new['activity'].value_counts().to_dict()
act_freq_dict_json = json.dumps(act_freq_dict, indent=4, ensure_ascii=False)
print(act_freq_dict_json)

{
    "Check for completeness": 7679,
    "Request info": 3883,
    "info received": 3833,
    "Perform checks": 3756,
    "Make decision": 3743,
    "EVENT 13 END": 2810,
    "New online application received": 2799,
    "notify reject": 1914,
    "Notify accept": 1859,
    "Deliver card": 1818,
    "review request received": 949,
    "time out": 930,
    "check for completeness": 655,
    "Info received": 361,
    "perform checks": 333,
    "make decision": 317,
    "request info": 311,
    "new online application received": 265,
    "eVENT 13 END": 243,
    "deliver card": 155,
    "notify accept": 148,
    "Notify reject": 147,
    "Time out": 86,
    "Review request received": 76,
    "Check for completenes": 61,
    "Requets info": 55,
    "Check for ocmpleteness": 50,
    "info erceived": 47,
    "Perfor mchecks": 47,
    "Make edcision": 44,
    "equest info": 42,
    "Check for complteeness": 42,
    "Check for completness": 39,
    "Check for comleteness": 39,
    "Chek for co

In [4]:
SYSTEM_PROMPT_STEP1 = """
You are an expert Process Mining Data Pre-processor.
Your goal is to filter a raw list of activity names based on specific criteria provided in the User Prompt.

### KNOWLEDGE BASE: IMPERFECTION PATTERNS
Use these definitions to identify which labels belong to which category.

1.  **Polluted Labels (Mutable Qualifiers):**
    Labels that share a immutable boiler-plate text but differ due to mutable text (e.g., embedded IDs or codes).
    * **Detection Criteria:**
        * **Long Numeric IDs:** 8+ digits (e.g., `20260122`, `9988776655`).
        * **Mixed Codes:** 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
        * **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

2.  **Distorted Labels (Character-Level Corruption):**
    Labels containing specific character-level corruptions (typos, OCR faults) of a canonical form. Unlike synonyms, these are "Noise".
    * **Detection Criteria:**
        1.  **Case Mutation:** Identical spelling, different capitalization (e.g., "Open" vs "open" vs "OPEN").
        2.  **Character Omission:** Exactly ONE missing character (e.g., "Invoce" vs "Invoice").
        3.  **Character Insertion:** Exactly ONE extra character (e.g., "Innvoice" vs "Invoice").
        4.  **Character Transposition:** Two adjacent characters swapped (e.g., "Ivnoice" vs "Invoice").
        5.  **Keyboard Proximity:** Exactly ONE character substituted by a QWERTY neighbor (e.g., "Invoicr" vs "Invoice").

3. **Synonymous Labels (Semantic Equivalence):**
   Labels that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step. 
   * **Detection Criteria (Ontology Rules):**
        1. **Linguistic & Domain Synonyms:** Different words representing the same concept within the process context (e.g., "Ship Item" vs "Dispatch Goods", "DrSeen" vs "Medical Assign").
        2. **Phrase Variation (Verb/Object Shift):** Labels sharing a core component (usually the Object) while using synonymous verbs or adjectives (e.g., "Create Invoice" vs "Generate Invoice", "Start instance" vs "Start process").
        3. **Grammatical Transformation:** Changing parts of speech (Noun ↔ Verb) or sentence structure while retaining the core meaning (e.g., "Give approval" vs "Approve", "Conduct analysis" vs "Analyze").
        4. **Containment & Refinement:** One label is a concise or verbose version of the other, often omitting non-essential adjectives, prepositions, or 'online/offline' qualifiers (e.g., "Receive signed contract" vs "Receive contract", "Register for course" vs "Register course").

### GLOBAL INSTRUCTION
- **Role:** Function as a logic engine. Do not assume all imperfections exist.
- **Priority:** The strict filtering logic in the **User Prompt** overrides general definitions here.
- **OUTPUT FORMAT:** Always return valid JSON as requested by the User Prompt.

"""

USER_PROMPT_DISTORTED_STEP1 = f"""
### TASK: Identify Canonical 'Clean Labels' for Distorted Clusters

**OBJECTIVE:**
Analyze the provided **INPUT DATA (Activity Frequencies)** to detect "Distorted Label" clusters.
For each cluster, determine the single **Canonical (Clean) Label**.

**STRICT EXECUTION STEPS:**

1. **Detect Distortion Clusters:**
   * Group labels that are character-level variations based on the System Prompt's 'Distorted Labels' criteria (Case Mutation, Omission, Insertion, Transposition, Keyboard Proximity).
   * **CRITICAL RULE (Anti-Acronym Bias):** Do NOT assume all-uppercase labels are valid acronymsor proper nouns.
     * Treat strings like "CHCEK", "EVNET", "PROCES" as potential typos of "Check", "Event", "Process".
     * Even if a word is ALL CAPS, checks for transposition/omission/insertion MUST be applied equally.
     * *Example:* Group `["Check", "CHCEK", "check"]` together.

2. **Select Canonical (Clean) Label:**
   * For each cluster, identify the **One True Clean Label** using this hierarchy:
     * **Rule A (Spelling Correction ONLY):** If it is a clear typo (e.g., "Logn" vs "Login"), choose the linguistically correct word.
     * **Rule B (Case Mutation Handling - PRIORITY):** If the difference is **ONLY CAPITALIZATION** (e.g., "login" vs "Login" vs "LOGIN"), **IGNORE grammar rules.**
       * **YOU MUST SELECT THE LABEL WITH THE HIGHEST FREQUENCY.**
       * Do not choose "Login" just because it looks proper. If "login" count > "Login" count, pick "login".

3. **Filter & Output:**
   * Collect **ONLY** the selected Canonical Clean Labels from Step 2.
   * **Discard** distorted variants and isolated unique labels.

**INPUT DATA (Label Frequencies):**
{act_freq_dict_json}

***OUTPUT FORMAT GUIDELINES***
Return a JSON Object with two keys:
1. "found": Boolean.
2. "data": List of strings.

**CRITICAL FORMATTING CONSTRAINT:**
- **PRESERVE EXACT CASING:** Return the string **EXACTLY** as it appears in the `INPUT DATA`.
- **DO NOT** auto-capitalize (e.g., do not turn "system check" into "System Check").

**Example Scenario (Using Dummy Data):**
* **Input:** `{{"login": 5000, "Login": 100, "Logn": 5, "System Check": 200}}`
* **Analysis:**
   * Cluster 1: `["login", "Login", "Logn"]`
     - "Logn" is a typo -> Discard.
     - "login" (5000) vs "Login" (100) -> "login" has higher frequency. **Select "login"**.
   * "System Check" has no variants -> Discard.
* **Output:** `{{"found": true, "data": ["login"]}}`

**CONSTRAINT:**
- Output **ONLY** the JSON object.
"""

In [5]:
prompt = [{"role": "system", "content": SYSTEM_PROMPT_STEP1},
          {"role":"user","content": USER_PROMPT_DISTORTED_STEP1}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
distorted_list_json = json.dumps(test_output['data'], indent=4, ensure_ascii=False)


In [6]:
for k, v in test_output.items():
    print(f"{k}:")
    if isinstance(v, list):  # 값이 리스트인 경우
        if v:  # 리스트가 비어 있지 않을 때
            print(*v, sep="\n")
        else:  # 리스트가 비어 있을 때
            print(" (empty list)")
    else:  # 'found'와 같이 리스트가 아닌 경우
        print(f" {v}")
    print() # 가독성을 위한 한 줄 띄우기

found:
 True

data:
Check for completeness
Request info
info received
Perform checks
Make decision
EVENT 13 END
New online application received
notify reject
Notify accept
Deliver card
review request received
time out



In [7]:
distorted_answer =['Check for completeness', 'New online application received',
                   'Perform checks', 'Make decision', 'notify reject', 'time out',
                   'EVENT 13 END', 'Request info', 'info received',
                   'review request received', 'Notify accept', 'Deliver card']
predicted_labels = test_output['data']


pred_set = set(predicted_labels)
true_set = set(distorted_answer)

correct_matches = pred_set & true_set 
wrong_predictions = pred_set - true_set
missed_labels = true_set - pred_set

precision = len(correct_matches) / len(pred_set) if len(pred_set) > 0 else 0.0
recall = len(correct_matches) / len(true_set) if len(true_set) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"📊 [Evaluation Report]")
print(f"-" * 40)
print(f"✅ Accuracy Metrics:")
print(f"   - Precision: {precision:.2%}  (Correct / Total Predicted)")
print(f"   - Recall:    {recall:.2%}  (Correct / Total Actual)")
print(f"   - F1 Score:  {f1_score:.2f}")
print(f"-" * 40)

print(f"\n🔴 False Positives (Predicted but NOT in Ground Truth): {len(wrong_predictions)}")
if wrong_predictions:
    for label in sorted(list(wrong_predictions)):
        print(f"   FP -> '{label}'")

print(f"\n🟡 False Negatives (In Ground Truth but NOT Predicted): {len(missed_labels)}")
if missed_labels:
    for label in sorted(list(missed_labels)):
        print(f"   FN -> '{label}'")


📊 [Evaluation Report]
----------------------------------------
✅ Accuracy Metrics:
   - Precision: 100.00%  (Correct / Total Predicted)
   - Recall:    100.00%  (Correct / Total Actual)
   - F1 Score:  1.00
----------------------------------------

🔴 False Positives (Predicted but NOT in Ground Truth): 0

🟡 False Negatives (In Ground Truth but NOT Predicted): 0


In [9]:
SYSTEM_PROMPT_DISTORTED_STEP2 = """
You are an expert Data Quality Analyst specializing in **Typo Detection and OCR Correction**.
Your task is to identify specific events in an Event Log where the `activity` label is a **Distorted Version** of a provided list of "Clean Labels".

### 1. REFERENCE DATA
You will be provided with a list of **Clean Labels**. These are the correct, canonical forms.

### 2. DISTORTION CRITERIA (The "Noise" Rules)
Compare each log activity against the Clean Labels. A match counts ONLY if it meets one of these specific corruptions:
1. **Case Mutation:** Same spelling, different capitalization (e.g., "Check" vs "check").
2. **Character Omission:** Missing exactly 1 char (e.g., "Check" vs "Chek").
3. **Character Insertion:** Extra 1 char (e.g., "Check" vs "Checck").
4. **Character Transposition:** Swapped adjacent chars (e.g., "Check" vs "Chekc").
5. **Keyboard Proximity:** 1 char replaced by a nearby key (e.g., "Check" vs "Checl").

### 3. EXCLUSION RULES
* **Exact Match:** If the log activity matches a Clean Label exactly (byte-for-byte), **IGNORE IT**. We only want the errors.
* **Unrelated:** If the activity is completely different (e.g., "Check" vs "Approve"), ignore it.

### OUTPUT FORMAT
Return strictly a valid JSON object containing a single key "event_id" with a list of strings.
No markdown, no explanations.

**Example Structure:**
{
  "event_id": ["id_typo_1", "id_typo_2"]
}
"""
def get_distorted_user_prompt_step2(clean_labels, event_log_chunk):    
    return f"""
### TASK: Detect Distorted Labels

**OBJECTIVE:**
Scan the **Target Event Log**. Identify `event_id`s where the `activity` is a **typo/distortion** of one of the **Clean Labels**.

**1. CLEAN LABELS (Reference Standards):**
{clean_labels}

**2. TARGET EVENT LOG:**
{event_log_chunk}

***INSTRUCTIONS***
For each event in the log:
1. Compare its `activity` to the Clean Labels list.
2. Is it an **Exact Match**? -> **SKIP** (It's clean).
3. Is it a **Distorted Match** (Typo, Case, missing char)? -> **KEEP `event_id`**.
4. Is it **Completely Different**? -> **SKIP**.

***OUTPUT***
Return ONLY the JSON object with the list of detected event IDs.
"""

target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)
all_predicted_ids = set() 
for case in target_cases:
    try:
        USER_PROMPT = get_distorted_user_prompt_step2(distorted_list_json, case)
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT_DISTORTED_STEP2},
            {"role": "user", "content": USER_PROMPT}
        ]
        test_output = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
        if isinstance(test_output, dict):
            current_ids = test_output.get('event_id', [])
            all_predicted_ids.update(str(x) for x in current_ids)
            
    except Exception as e:
        print(f"Error processing a case: {e}")
        continue



In [13]:
target_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)]
ground_truth_df = target_df[target_df['label'].notna()]
actual_ids = set(ground_truth_df['event_id'].astype(str))
intersection = all_predicted_ids.intersection(actual_ids)
tp = len(intersection)
n_actual = len(actual_ids) 
n_pred = len(all_predicted_ids) 
recall = (tp / n_actual * 100) if n_actual > 0 else 0.0
precision = (tp / n_pred * 100) if n_pred > 0 else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print("\n" + "=" * 40)
print(f"   BATCH EVALUATION ({len(target_case_ids)} CASES)   ")
print("=" * 40)
print(f"1. Total Ground Truth (Actual) : {n_actual}")
print(f"2. Total Predictions (Model)   : {n_pred}")
print(f"3. True Positives (Matches)    : {tp}")
print("-" * 40)
print(f"▶ Recall    : {recall:.2f}%")
print(f"▶ Precision : {precision:.2f}%")
print(f"▶ F1 Score  : {f1_score:.2f}")
print("=" * 40)


   BATCH EVALUATION (200 CASES)   
1. Total Ground Truth (Actual) : 743
2. Total Predictions (Model)   : 746
3. True Positives (Matches)    : 701
----------------------------------------
▶ Recall    : 94.35%
▶ Precision : 93.97%
▶ F1 Score  : 94.16
